In [ ]:
# !pip install ucimlrepo tensorflow   # descomente se necessário
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
bean = fetch_ucirepo(id=602)
df = bean.data.original.copy()
print(df.shape)
df['Class'].value_counts()

In [ ]:
df['Class'].value_counts(normalize=True).round(3)

Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64

In [ ]:
# Transformação da classe em coluna numérica
df['rotulo'] = df['Class'].astype('category').cat.codes
classes = dict(df[['rotulo', 'Class']].drop_duplicates().sort_values('rotulo').values)
print(classes)

df_encoded = df.drop('Class', axis=1)
df_encoded.describe().T

Class
DERMASON    0.261
SIRA        0.194
SEKER       0.149
HOROZ       0.142
CALI        0.120
BARBUNYA    0.097
BOMBAY      0.038
Name: proportion, dtype: float64

In [ ]:
print("Linhas duplicadas:", df_encoded.duplicated().sum())
# Remoção das duplicadas (atribuindo o resultado e descartando o índice antigo)
df_encoded = df_encoded.drop_duplicates().reset_index(drop=True)
print("Shape após remoção:", df_encoded.shape)

feats = df_encoded.columns.drop('rotulo')
X = df_encoded[feats].values.astype('float32')
Y = df_encoded['rotulo'].values
n_features, n_classes = X.shape[1], len(classes)
print(f"{n_features} atributos, {n_classes} classes")

{np.int8(5): 'SEKER', np.int8(0): 'BARBUNYA', np.int8(1): 'BOMBAY', np.int8(2): 'CALI', np.int8(4): 'HOROZ', np.int8(6): 'SIRA', np.int8(3): 'DERMASON'}


In [ ]:
SPLIT_SEED = 42
X_trainfull, X_test, y_trainfull, y_test = train_test_split(
    X, Y, test_size=0.20, stratify=Y, random_state=SPLIT_SEED)   # X_test/y_test NÃO serão usados aqui

X_tr, X_val, y_tr, y_val = train_test_split(
    X_trainfull, y_trainfull, test_size=0.15, stratify=y_trainfull, random_state=SPLIT_SEED)

scaler = StandardScaler().fit(X_tr)
X_tr_s, X_val_s = scaler.transform(X_tr), scaler.transform(X_val)

# One-hot para a Entropia Cruzada Categórica
Y_tr  = keras.utils.to_categorical(y_tr,  n_classes)
Y_val = keras.utils.to_categorical(y_val, n_classes)

print(f"Treino: {X_tr_s.shape} | Validação: {X_val_s.shape} | Teste (reservado): {X_test.shape}")

,count,mean,std,min,25%,50%,75%,max
Area,13611.0,53048.284549,29324.095717,20420.000000,36328.000000,44652.000000,61332.000000,254616.000000
Perimeter,13611.0,855.283459,214.289696,524.736000,703.523500,794.941000,977.213000,1985.370000
MajorAxisLength,13611.0,320.141867,85.694186,183.601165,253.303633,296.883367,376.495012,738.860154
MinorAxisLength,13611.0,202.270714,44.970091,122.512653,175.848170,192.431733,217.031741,460.198497
AspectRatio,13611.0,1.583242,0.246678,1.024868,1.432307,1.551124,1.707109,2.430306
Eccentricity,13611.0,0.750895,0.092002,0.218951,0.715928,0.764441,0.810466,0.911423
ConvexArea,13611.0,53768.200206,29774.915817,20684.000000,36714.500000,45178.000000,62294.000000,263261.000000
EquivDiameter,13611.0,253.064220,59.177120,161.243764,215.068003,238.438026,279.446467,569.374358
Extent,13611.0,0.749733,0.049086,0.555315,0.718634,0.759859,0.786851,0.866195
Solidity,13611.0,0.987143,0.004660,0.919246,0.985670,0.988283,0.990013,0.994677


In [ ]:
CONFIG = dict(
    hidden_units=[64, 32],
    hidden_activation='relu',
    output_activation='softmax',
    learning_rate=0.01,
    momentum=0.0,
    epochs=100,
    batch_size=32,
)
SEEDS = [0, 1, 2, 3, 4]   # uma semente por treinamento -> 5 inicializações diferentes


def build_mlp(seed, cfg=CONFIG):
    """MLP com pesos inicializados a partir da semente informada."""
    model = keras.Sequential([keras.Input(shape=(n_features,))])
    for i, units in enumerate(cfg['hidden_units']):
        model.add(layers.Dense(units, activation=cfg['hidden_activation'],
                               kernel_initializer=keras.initializers.HeNormal(seed=seed + i),
                               name=f'oculta_{i+1}'))
    model.add(layers.Dense(n_classes, activation=cfg['output_activation'],
                           kernel_initializer=keras.initializers.GlorotUniform(seed=seed + 100),
                           name='saida'))
    model.compile(
        optimizer=keras.optimizers.SGD(learning_rate=cfg['learning_rate'], momentum=cfg['momentum']),
        loss=keras.losses.CategoricalCrossentropy(),
        metrics=['accuracy'])
    return model

build_mlp(0).summary()

np.int64(68)

In [ ]:
def reset_seeds(seed):
    """Reinicia todos os geradores de números aleatórios (Python, NumPy, TensorFlow)."""
    keras.backend.clear_session()
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    keras.utils.set_random_seed(seed)

histories, init_weights, models, rows = {}, {}, {}, []

for seed in SEEDS:
    reset_seeds(seed)
    model = build_mlp(seed)
    init_weights[seed] = model.get_layer('oculta_1').get_weights()[0].ravel().copy()
    loss0, acc0 = model.evaluate(X_tr_s, Y_tr, verbose=0)          # antes de treinar

    h = model.fit(X_tr_s, Y_tr, validation_data=(X_val_s, Y_val),
                  epochs=CONFIG['epochs'], batch_size=CONFIG['batch_size'],
                  shuffle=True, verbose=0)
    hist = pd.DataFrame(h.history); hist.index += 1
    histories[seed], models[seed] = hist, model

    acc_tr = hist['accuracy']
    rows.append({
        'Semente': seed,
        'Perda inicial': loss0,
        'Acurácia inicial': acc0,
        'Perda final (treino)': hist['loss'].iloc[-1],
        'Acurácia final (treino)': acc_tr.iloc[-1],
        'Perda final (val)': hist['val_loss'].iloc[-1],
        'Acurácia final (val)': hist['val_accuracy'].iloc[-1],
        'Época p/ 90% (treino)': int(acc_tr.ge(0.90).idxmax()) if acc_tr.ge(0.90).any() else np.nan,
        'Melhor época (val_loss)': int(hist['val_loss'].idxmin()),
    })
    print(f"Semente {seed}: perda={hist['loss'].iloc[-1]:.4f}  acc={acc_tr.iloc[-1]:.4f}  "
          f"| val_perda={hist['val_loss'].iloc[-1]:.4f}  val_acc={hist['val_accuracy'].iloc[-1]:.4f}")

resultados = pd.DataFrame(rows).set_index('Semente')

In [ ]:
resumo = resultados.agg(['mean', 'std', 'min', 'max'])
resumo.index = ['Média', 'Desvio padrão', 'Mínimo', 'Máximo']
display(resultados)
display(resumo)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for seed in SEEDS:
    ax.hist(init_weights[seed], bins=40, histtype='step', linewidth=1.5, label=f'Semente {seed}')
ax.set(title='Pesos iniciais — camada oculta 1', xlabel='valor do peso', ylabel='frequência')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
cores = plt.cm.tab10.colors
for k, seed in enumerate(SEEDS):
    h = histories[seed]
    axes[0, 0].plot(h['loss'], color=cores[k], label=f'Semente {seed}')
    axes[0, 1].plot(h['accuracy'], color=cores[k], label=f'Semente {seed}')
    axes[1, 0].plot(h['loss'], color=cores[k], alpha=.35)
    axes[1, 0].plot(h['val_loss'], color=cores[k], linestyle='--')
    axes[1, 1].plot(h['accuracy'], color=cores[k], alpha=.35)
    axes[1, 1].plot(h['val_accuracy'], color=cores[k], linestyle='--')

axes[0, 0].set(title='Perda (treino)', ylabel='Entropia cruzada')
axes[0, 1].set(title='Acurácia (treino)', ylabel='Acurácia')
axes[1, 0].set(title='Perda: treino (claro) vs validação (tracejado)', xlabel='Época', ylabel='Entropia cruzada')
axes[1, 1].set(title='Acurácia: treino (claro) vs validação (tracejado)', xlabel='Época', ylabel='Acurácia')
for ax in axes.ravel(): ax.grid(alpha=.3)
axes[0, 0].legend(); axes[0, 1].legend(loc='lower right')
plt.tight_layout(); plt.show()

In [ ]:
# Média ± desvio padrão entre as 5 inicializações
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, met, titulo in [(axes[0], 'loss', 'Perda'), (axes[1], 'accuracy', 'Acurácia')]:
    for suf, cor, nome in [('', 'tab:blue', 'treino'), ('val_', 'tab:orange', 'validação')]:
        M = pd.concat([histories[s][suf + met] for s in SEEDS], axis=1)
        m, sd = M.mean(axis=1), M.std(axis=1)
        ax.plot(m, color=cor, label=f'{nome} (média)')
        ax.fill_between(m.index, m - sd, m + sd, color=cor, alpha=.25)
    ax.set(title=f'{titulo} — média ± desvio entre sementes', xlabel='Época'); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
melhor = resultados['Perda final (treino)'].idxmin()
pred = models[melhor].predict(X_tr_s, verbose=0).argmax(axis=1)
nomes = [classes[i] for i in range(n_classes)]
print(f"Semente {melhor}\n")
print(classification_report(y_tr, pred, target_names=nomes, digits=3))

cm = confusion_matrix(y_tr, pred, normalize='true')
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(n_classes), nomes, rotation=45, ha='right'); ax.set_yticks(range(n_classes), nomes)
for i in range(n_classes):
    for j in range(n_classes):
        ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center', color='white' if cm[i, j] > .5 else 'black', fontsize=8)
ax.set(xlabel='Previsto', ylabel='Real', title='Matriz de confusão normalizada (treino)')
plt.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
r, s = resultados, resumo
acc_tr, acc_val = r['Acurácia final (treino)'], r['Acurácia final (val)']
loss_tr, loss_val = r['Perda final (treino)'], r['Perda final (val)']
gap = (acc_tr - acc_val).mean()
gap_loss = (loss_val - loss_tr).mean()
cv_loss = loss_tr.std() / loss_tr.mean() * 100
amp_acc = (acc_tr.max() - acc_tr.min()) * 100
ep90 = r['Época p/ 90% (treino)']
best_ep = r['Melhor época (val_loss)']

# Diagnóstico de over/underfitting
if acc_tr.mean() < 0.85:
    diag = ("a acurácia de treino ainda é relativamente baixa, o que sugere **underfitting** — o modelo pode precisar de "
            "mais épocas, maior taxa de aprendizado ou mais capacidade.")
elif gap > 0.03 or gap_loss > 0.05:
    diag = (f"há uma diferença perceptível entre treino e validação (acurácia {gap*100:.2f} p.p. maior no treino), "
            "indicando **início de overfitting**; regularização ou parada antecipada seriam recomendáveis.")
else:
    diag = (f"treino e validação permanecem próximos (diferença média de {gap*100:.2f} p.p. na acurácia e "
            f"{gap_loss:+.4f} na perda), **sem indícios relevantes de overfitting**; como a acurácia de treino é alta, "
            "também **não há underfitting**.")

if best_ep.mean() < CONFIG['epochs'] * 0.8:
    obs_ep = (f"A menor perda de validação ocorreu, em média, na época {best_ep.mean():.0f}; depois disso a validação "
              "estabiliza ou piora levemente, sinal de que o ganho adicional vem principalmente do ajuste ao treino.")
else:
    obs_ep = (f"A perda de validação ainda caía nas últimas épocas (melhor época média: {best_ep.mean():.0f}), "
              "o que indica que o modelo não saturou e poderia se beneficiar de mais épocas.")

rec = pd.Series(np.diag(cm), index=nomes).sort_values()
piores = ', '.join(f"{c} ({v*100:.1f}%)" for c, v in rec.head(3).items())
melhor_cls = f"{rec.index[-1]} ({rec.iloc[-1]*100:.1f}%)"
sens = "baixa" if cv_loss < 5 and amp_acc < 1 else ("moderada" if cv_loss < 15 and amp_acc < 3 else "alta")

display(Markdown(f'''
**Resumo numérico (5 inicializações)**

| Métrica | Média ± desvio | Mín – Máx |
|---|---|---|
| Perda final (treino) | {loss_tr.mean():.4f} ± {loss_tr.std():.4f} | {loss_tr.min():.4f} – {loss_tr.max():.4f} |
| Acurácia final (treino) | {acc_tr.mean():.4f} ± {acc_tr.std():.4f} | {acc_tr.min():.4f} – {acc_tr.max():.4f} |
| Perda final (validação) | {loss_val.mean():.4f} ± {loss_val.std():.4f} | {loss_val.min():.4f} – {loss_val.max():.4f} |
| Acurácia final (validação) | {acc_val.mean():.4f} ± {acc_val.std():.4f} | {acc_val.min():.4f} – {acc_val.max():.4f} |
| Perda inicial (antes do treino) | {r['Perda inicial'].mean():.4f} ± {r['Perda inicial'].std():.4f} | {r['Perda inicial'].min():.4f} – {r['Perda inicial'].max():.4f} |
| Época em que atinge 90% (treino) | {ep90.mean():.1f} ± {ep90.std():.1f} | {ep90.min():.0f} – {ep90.max():.0f} |

### c. Análise e discussão
- **Comparação entre os treinamentos:** as cinco execuções partem de perdas iniciais diferentes
  ({r['Perda inicial'].min():.3f} a {r['Perda inicial'].max():.3f}; o valor de referência para palpite aleatório é
  ln(7) ≈ {np.log(n_classes):.3f}), mas convergem para perdas finais de {loss_tr.min():.4f} a {loss_tr.max():.4f}.
  A acurácia de 90% no treino é atingida entre as épocas {ep90.min():.0f} e {ep90.max():.0f}, ou seja, a inicialização
  afeta sobretudo a **velocidade inicial** de convergência, e pouco o ponto final.
- **Estabilidade e desempenho:** coeficiente de variação da perda final = {cv_loss:.2f}% e amplitude da acurácia final
  de treino = {amp_acc:.2f} p.p. Nenhuma execução divergiu ou ficou presa em platô.
- **Underfitting / overfitting:** {diag} {obs_ep}
- **Sensibilidade à inicialização:** **{sens}**. Com He Normal + ReLU e dados padronizados, os pontos de partida
  diferentes levam a mínimos de qualidade equivalente.
- **Comportamento geral:** a perda cai rapidamente nas primeiras épocas e depois decresce de forma lenta e suave,
  padrão esperado do SGD com taxa fixa. Pela matriz de confusão, as classes com menor recall no treino são
  {piores} — em geral variedades de formato parecido —, enquanto {melhor_cls} é a mais bem separada.

### d. Conclusão da etapa
- O modelo **foi capaz de aprender os padrões básicos** das 7 classes, com acurácia média de treino de
  {acc_tr.mean()*100:.2f}% e validação de {acc_val.mean()*100:.2f}%.
- **ReLU + Softmax** se mostraram adequadas: convergência sem saturação e saídas probabilísticas coerentes com a
  entropia cruzada categórica.
- O treinamento é **estável** frente à inicialização dos pesos (variação {sens}), de modo que a topologia
  16-64-32-7 fica validada como base para a **Questão 2** (efeito da taxa de aprendizado e do momento).
'''))